<a href="https://colab.research.google.com/github/boredanon/RushHourClusterDataset/blob/main/RushHour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cloning the data

In [2]:
!git clone https://github.com/boredanon/RushHourDataset.git
!sudo apt-get update && sudo apt-get install -y libboost-all-dev
!g++ -Wall -Wextra -O3 RushHourDataset/retrograde_analysis/*.cpp -o RushHourDataset/retrograde_analysis/main -lboost_program_options

Cloning into 'RushHourDataset'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 36 (delta 2), reused 35 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 175.97 KiB | 703.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.
Filtering content: 100% (12/12), 7.66 GiB | 42.32 MiB/s, done.


Functions for use in the analysis.

In [1]:
import numpy as np
import polars as pl
from enum import Enum

class CountMode(Enum):
    TOTAL = 0
    TWOBYONE = 1
    THREEBYONE = 2

def unrank(ranks, base=6, dims=12):
    ranks = np.array(ranks, dtype=np.int64)
    powers = base ** np.arange(dims-1,-1,-1)
    unranked = (ranks[:, None] // powers) % base
    return unranked

def _finalize_if_lazy(df_or_lf: pl.DataFrame | pl.LazyFrame) -> pl.DataFrame:
    if isinstance(df_or_lf, pl.LazyFrame):
        return (
            df_or_lf
            .group_by(pl.all())
            .agg(pl.len())
            .collect(engine="cpu")
        )
    return df_or_lf

def _get_working_column(cluster_col: pl.Expr) -> pl.Expr:
    string_col = cluster_col.cast(pl.String)
    clean_str = string_col.str.replace("0x", "", literal=True)
    hex_parsed = clean_str.str.to_integer(base=16, strict=False)
    return hex_parsed.fill_null(string_col.str.to_integer(base=10, strict=False))

def _calculate_piece_count(cluster_col: pl.Expr, mode : CountMode) -> pl.Expr:
    base = 6
    dims = 12
    dim_exprs = []

    working_col = _get_working_column(cluster_col)

    for i in range(dims):
        power = base ** (dims - 1 - i)
        dim_expr = (working_col // power) % base
        dim_exprs.append(dim_expr)

    count_1 = pl.sum_horizontal([expr == 1 for expr in dim_exprs])
    count_2 = pl.sum_horizontal([expr == 2 for expr in dim_exprs])
    count_3 = pl.sum_horizontal([expr == 3 for expr in dim_exprs])
    count_4 = pl.sum_horizontal([expr == 4 for expr in dim_exprs])
    count_5 = pl.sum_horizontal([expr == 5 for expr in dim_exprs])

    p_3x1 = count_1 + count_2 + count_3
    p_2x1 = count_2 + count_3 + count_4 + (2*count_5)

    if mode == CountMode.THREEBYONE:
        return p_3x1
    elif mode == CountMode.TWOBYONE:
        return p_2x1
    else:
        return p_2x1 + p_3x1


def cluster_filter_by_size(df_or_lf : pl.DataFrame | pl.LazyFrame, min : int, max : int) -> pl.DataFrame:
    result = df_or_lf.filter(pl.col("Cluster Size").is_between(min, max))
    return _finalize_if_lazy(result)

def cluster_filter_by_length(df_or_lf : pl.DataFrame | pl.LazyFrame, min : int, max : int) -> pl.DataFrame:
    result = df_or_lf.filter(pl.col("Max Length").is_between(min, max))
    return _finalize_if_lazy(result)

def cluster_filter_by_pieces(df_or_lf : pl.DataFrame | pl.LazyFrame, mode : CountMode, target_count : int) -> pl.DataFrame:
    piece_count_expr = _calculate_piece_count(pl.col("Cluster"), mode)
    result = df_or_lf.filter(piece_count_expr == target_count)
    return _finalize_if_lazy(result)

## Available, but not recommended due to memory constraints.
def cluster_filter_by_legal(df_or_lf : pl.DataFrame | pl.LazyFrame) -> pl.DataFrame:
    result = df_or_lf.filter(pl.col("Legal") == 1)
    return _finalize_if_lazy(result)

def cluster_filter_by_has_solution(df_or_lf : pl.DataFrame | pl.LazyFrame) -> pl.DataFrame:
    result = df_or_lf.filter(pl.col("Has Solution") == 1)
    return _finalize_if_lazy(result)

def transform_clusters_for_input(df : pl.Series):
    base = 6
    if df.dtype == pl.String:
        df = df.str.replace("0x", "", literal=True).str.to_integer(base=16)
    ranks = df.to_numpy()
    powers = base ** np.arange(11, -1, -1, dtype=np.uint64)
    matrix = (ranks[:, None] // powers) % base
    matrix = np.delete(matrix, 8, axis=1)
    new_powers = base ** np.arange(10, -1, -1, dtype=np.uint64)
    new_ranks = np.dot(matrix, new_powers)
    new_ranks = new_ranks.astype(int).tolist()
    new_ranks = [ hex(int(i)) for i in new_ranks]
    return new_ranks

## DataFrame manipulation.

In [2]:
lazy_df = pl.scan_csv("RushHourDataset/part*.txt")
# It is recommended to use a query that highly reduces the size of the dataset first.
# For example: by length of cluster size will always result in a legal cluster.
# Filtering first by piece can generate a DataFrame too large to manipulate.
df = cluster_filter_by_length(lazy_df, 0, 10)
df = cluster_filter_by_pieces(df, CountMode.TOTAL, 4)
print(df)

shape: (1_934, 7)
┌────────────┬───────┬──────────────┬──────────────┬────────────┬───────┬─────┐
│ Cluster    ┆ Legal ┆ Has Solution ┆ Cluster Size ┆ Max Length ┆ Time  ┆ len │
│ ---        ┆ ---   ┆ ---          ┆ ---          ┆ ---        ┆ ---   ┆ --- │
│ str        ┆ i64   ┆ i64          ┆ i64          ┆ i64        ┆ f64   ┆ u32 │
╞════════════╪═══════╪══════════════╪══════════════╪════════════╪═══════╪═════╡
│ 0x4346df0  ┆ 1     ┆ 1            ┆ 102          ┆ 3          ┆ 0.0   ┆ 1   │
│ 0x28135e0  ┆ 1     ┆ 1            ┆ 249          ┆ 3          ┆ 0.001 ┆ 1   │
│ 0x111988   ┆ 1     ┆ 1            ┆ 300          ┆ 2          ┆ 0.0   ┆ 1   │
│ 0x1338f84  ┆ 1     ┆ 1            ┆ 11           ┆ 2          ┆ 0.0   ┆ 1   │
│ 0x601d3a0  ┆ 1     ┆ 1            ┆ 212          ┆ 2          ┆ 0.0   ┆ 1   │
│ …          ┆ …     ┆ …            ┆ …            ┆ …          ┆ …     ┆ …   │
│ 0x1410c0   ┆ 1     ┆ 1            ┆ 334          ┆ 3          ┆ 0.001 ┆ 1   │
│ 0x112da6   ┆ 1     ┆

In [28]:
import subprocess
import os
list = transform_clusters_for_input(df["Cluster"])

with open("temp.txt", "w") as file:
    for cluster in list:
        print(cluster, file=file)

# Corrected path to include RushHourDataset directory
os.chmod("./RushHourDataset/retrograde_analysis/main", 0o755)

binary_path = "./RushHourDataset/retrograde_analysis/main -f temp.txt -b 1 -a retro -m length > output_boards_length.txt"
result = subprocess.run(binary_path, shell=True)

binary_path = "./RushHourDataset/retrograde_analysis/main -f temp.txt -b 1 -a retro -m entropy > output_boards_entropy.txt"
result = subprocess.run(binary_path, shell=True)

binary_path = "./RushHourDataset/retrograde_analysis/main -f temp.txt -b 1 -a retro -m e-l > output_boards_e-l.txt"
result = subprocess.run(binary_path, shell=True)

os.remove("temp.txt")